# Deep Dive: Parallelization

## 📋 Summary

**Parallelization** runs several **independent** LLM calls at the same time and then **combines** their results.
It is the fan-out / fan-in pattern:

```
              ┌──▶ Methods Reviewer ────┐
input ──▶ FAN-OUT ──▶ Statistics Reviewer ──▶ FAN-IN (Meta-reviewer) ──▶ report
              └──▶ Ethics Reviewer ─────┘
```

Two flavors, both in this notebook:

| Flavor | What runs in parallel | Example |
|---|---|---|
| **Sectioning** | *Different* agents on *different aspects* of the same input | A methods, a statistics and an ethics reviewer read the same paper |
| **Voting** | The *same* agent several times, then a majority decides | Ask 5 times "does this study have a control group?" |

**Why it works:** an LLM call spends almost all its time **waiting on the network**. Threads let those waits overlap, so
three 3-second calls take about 3 seconds in total, not 9. Parallelization gives **speed** (sectioning) or **confidence** (voting).
It only works when the calls are **independent**: if agent B needs agent A's output, that is a chain, not a fan-out.

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | Sequential baseline (and why threads help) | 🟢 Beginner |
| 2 | `threading.Thread` with a shared dict | 🟡 Intermediate |
| 3 | `ThreadPoolExecutor`: the cleaner tool | 🟡 Intermediate |
| 4 | Fan-in: an aggregator agent | 🟡 Intermediate |
| 5 | Voting: same agent, many runs | 🔴 Advanced |
| 6 | Failure modes: silent thread errors, rate limits | 🔴 Advanced |
| 7 | Exercises | 🏋️ Practice |

**How to use this notebook:** run the cells in order. Cells marked `# TODO` are for you to fill in.
Timings will vary with your connection and the API load.

## 0. Setup

In [ ]:
# Key input: in Colab, add OPENAI_API_KEY in the 🔑 Secrets panel (left sidebar).
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [ ]:
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4.1-nano"  # cheap model, plenty for these experiments


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))

## 1. Sequential Baseline 🟢

**Running example:** three **peer reviewers** each read the same research abstract and comment on a different aspect. They do not need each other's
opinion, so they are independent.

First, a free simulation (no API) so you can *see* why threads help: each fake agent "waits" one second.

In [ ]:
def fake_agent(name: str) -> str:
    time.sleep(1)          # stands in for a network call
    return f"{name} done"


names = ["Methods Reviewer", "Statistics Reviewer", "Ethics Reviewer"]

start = time.perf_counter()
_ = [fake_agent(n) for n in names]
print(f"Sequential: {time.perf_counter() - start:.1f}s")

start = time.perf_counter()
with ThreadPoolExecutor() as ex:
    _ = list(ex.map(fake_agent, names))
print(f"Parallel  : {time.perf_counter() - start:.1f}s")

Now the real reviewers. Each one has a **narrow remit**, which is what makes the sectioning pattern work.

In [ ]:
ABSTRACT = """
We surveyed 48 volunteers about a new sleep-tracking app. After 4 weeks, 39 reported better sleep, so we conclude the app
improves sleep quality (p = 0.03). Participants were recruited from the authors' own social media followers. No control group
was used. Sleep data were self-reported and participants' names and locations were included in the supplementary data.
"""

REVIEWERS = {
    "Methods Reviewer": (
        "You are a peer reviewer focused ONLY on research methodology: study design, sampling, controls, bias. "
        "Reply in at most 50 words and end with 'VERDICT: sound', 'VERDICT: concerns' or 'VERDICT: flawed'."
    ),
    "Statistics Reviewer": (
        "You are a peer reviewer focused ONLY on statistics: sample size, tests, p-values, whether the conclusion follows from the numbers. "
        "Reply in at most 50 words and end with 'VERDICT: sound', 'VERDICT: concerns' or 'VERDICT: flawed'."
    ),
    "Ethics Reviewer": (
        "You are a peer reviewer focused ONLY on research ethics: consent, privacy, data protection. "
        "Reply in at most 50 words and end with 'VERDICT: sound', 'VERDICT: concerns' or 'VERDICT: flawed'."
    ),
}


def run_reviewer(name: str, abstract: str) -> str:
    return call_llm(REVIEWERS[name], f"Abstract:\n{abstract}")


start = time.perf_counter()
sequential_reviews = {n: run_reviewer(n, ABSTRACT) for n in REVIEWERS}
sequential_time = time.perf_counter() - start
print(f"Sequential real calls: {sequential_time:.1f}s")
for n, r in sequential_reviews.items():
    show(n, r)

### 🧐 Observation
- Would the Ethics Reviewer's answer change if it had seen the Methods Reviewer's answer first? If not, the calls are independent and safe to parallelize.
- Why are threads (not multiple processes) enough for API calls? (Hint: what is the CPU doing while a request is in flight?)

## 2. `threading.Thread` with a Shared Dict 🟡

The classic approach: start one **thread per agent**; each writes its result into a **shared dict** under its own key; then `join()` all threads
so the main program waits until every one has finished.

Writing to *different keys* of a dict from different threads is safe in CPython. If threads had to **read-modify-write the same value** (for example, a counter), you would need a `threading.Lock`.

In [ ]:
agent_outputs = {}          # shared between threads


def worker(name: str, abstract: str):
    agent_outputs[name] = run_reviewer(name, abstract)


start = time.perf_counter()
threads = [threading.Thread(target=worker, args=(n, ABSTRACT)) for n in REVIEWERS]
for t in threads:
    t.start()
for t in threads:
    t.join()                # wait for all of them
thread_time = time.perf_counter() - start

print(f"Threaded: {thread_time:.1f}s  (sequential was {sequential_time:.1f}s)")
print("Keys collected:", list(agent_outputs))

### 🧐 Observation
- Results arrive in **whatever order the calls finish**. Why do we key the dict by agent name instead of appending to a list?
- What is missing from `agent_outputs` if one of the threads crashes? (Section 6 shows this.)

## 3. `ThreadPoolExecutor`: the Cleaner Tool 🟡

`concurrent.futures.ThreadPoolExecutor` gives you what you had to build by hand:
- a **pool with a maximum number of workers** (protects you from rate limits),
- **futures**: objects that hold the result *or the exception* of each call,
- `as_completed` to process results as soon as each one is ready.

We wrap it in a small reusable helper that returns `(results, errors)`.

In [ ]:
def run_parallel(fn, items, max_workers: int = 4):
    """Run fn(item) for every item in parallel. Returns (results, errors) dicts keyed by item."""
    results, errors = {}, {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fn, item): item for item in items}
        for fut in as_completed(futures):
            item = futures[fut]
            try:
                results[item] = fut.result()      # re-raises the worker's exception, if any
            except Exception as e:
                errors[item] = e
    return results, errors


start = time.perf_counter()
reviews, errors = run_parallel(lambda n: run_reviewer(n, ABSTRACT), list(REVIEWERS))
print(f"ThreadPoolExecutor: {time.perf_counter() - start:.1f}s | errors: {errors}")
for n, r in reviews.items():
    show(n, r)

## 4. Fan-in: an Aggregator Agent 🟡

Parallel outputs are only useful once they are **combined**. The **fan-in** step is usually another LLM agent that reads all the results
and produces one answer. Give it a job that is **different** from the reviewers' (compare, weigh, decide), otherwise it just repeats them.

In [ ]:
def meta_reviewer(abstract: str, reviews: dict) -> str:
    system = """You are the handling editor. You receive an abstract and three specialist reviews.
Write a decision in under 100 words:
1. Decision: accept / major revision / reject
2. The single most serious problem
3. Where the reviewers agree and where they disagree"""
    body = "\n\n".join(f"### {name}\n{text}" for name, text in reviews.items())
    return call_llm(system, f"Abstract:\n{abstract}\n\nReviews:\n{body}")


show("Editor's decision", meta_reviewer(ABSTRACT, reviews))

### 🧐 Observation
- What would happen to the final decision if the Ethics Reviewer had **failed** and its review were missing? Should the aggregator be told?
- Fan-in adds one **sequential** call at the end. Where does the total time now come from?

## 5. Voting: Same Agent, Many Runs 🔴

The second flavor: ask **the same question several times** and take the **majority** answer. With a temperature above 0 the model samples different
reasoning paths, so agreement between runs is a cheap **confidence signal**.
Use it for **classification-like** questions with a small set of possible answers.

In [ ]:
QUESTION = "Does this study report using a control group? Answer with exactly one word: yes or no."


def vote_once(_i: int) -> str:
    answer = call_llm("You are a careful research auditor.", f"{QUESTION}\n\nAbstract:\n{ABSTRACT}", temperature=0.9)
    return answer.lower().strip(" .!\n")


N_VOTES = 5
votes, vote_errors = run_parallel(vote_once, list(range(N_VOTES)), max_workers=N_VOTES)
tally = Counter(votes.values())
winner, count = tally.most_common(1)[0]
print("Votes:", dict(tally))
print(f"Winner: {winner!r} with agreement {count / len(votes):.0%}")

### 🧐 Observation
- What agreement level would make you **trust** the answer, and which would send it to a human?
- Voting costs N times as much. When is that worth it, and when is a single temperature-0 call enough?

## 6. Failure Modes: Silent Thread Errors and Rate Limits 🔴

Parallel code fails differently from sequential code.

| Problem | Symptom | Fix |
|---|---|---|
| **Exception inside a raw `Thread`** | Traceback printed, main program **continues** with a missing result | Use futures (`fut.result()` re-raises), or catch inside the worker |
| **Rate limits (HTTP 429)** | Random failures when many calls fire at once | Limit `max_workers`, retry with **exponential backoff** |
| **Non-deterministic order** | Results in a different order each run | Key by name, sort before display |
| **Shared mutable state** | Corrupted counters or lists | Write to separate keys, or protect with `threading.Lock` |

Free demo (no API): a reviewer that **fails on purpose**.

In [ ]:
def flaky_reviewer(name: str) -> str:
    if name == "Ethics Reviewer":
        raise RuntimeError("simulated API failure")
    return f"{name}: ok"


# 1) raw threads: the error does not reach the main program
collected = {}


def raw_worker(name: str):
    collected[name] = flaky_reviewer(name)


threads = [threading.Thread(target=raw_worker, args=(n,)) for n in REVIEWERS]
for t in threads:
    t.start()
for t in threads:
    t.join()
print("\nRaw threads collected:", list(collected), " <- Ethics Reviewer silently missing")

# 2) futures: the error is captured and reported
ok, failed = run_parallel(flaky_reviewer, list(REVIEWERS))
print("Executor results:", list(ok))
print("Executor errors :", {k: str(v) for k, v in failed.items()})

## 7. Exercises 🏋️

### Exercise 1: Parallel product-page audit
**Domain:** an online shop wants each product page audited before launch.
Build three parallel reviewers (**SEO**, **Accessibility**, **Tone of voice**) plus a fan-in agent that produces a prioritized fix list (top 3 issues).
Use `run_parallel` and keep every reviewer's remit narrow.

In [ ]:
PRODUCT_PAGE = """
Title: Best Backpack Ever!!!
Description: buy now buy now the backpack is very good and cheap, waterproof, best backpack, backpack for school and travel backpack.
Image: backpack.jpg (no alt text)
Button: Click here
"""

AUDITORS = {
    # TODO: "SEO Auditor": "system prompt...", "Accessibility Auditor": "...", "Tone Auditor": "..."
}


def run_auditor(name: str, page: str) -> str:
    # TODO: call call_llm with the auditor's system prompt
    pass


def summarize_audit(page: str, audits: dict) -> str:
    # TODO: fan-in agent that returns the 3 most important fixes
    pass


# TODO: run the auditors in parallel with run_parallel, then call summarize_audit

### Exercise 2: Retries with exponential backoff
Extend `run_parallel` into `run_parallel_with_retry(fn, items, max_workers, retries=3)`.
When a call raises, wait `1s, 2s, 4s...` (plus a little random jitter) and try again, then report the items that still failed.

Design question: why should the **wait happen inside the worker** and not in the main thread?

In [ ]:
import random


def run_parallel_with_retry(fn, items, max_workers: int = 4, retries: int = 3):
    def with_retry(item):
        # TODO: try fn(item); on failure sleep with exponential backoff + jitter and retry
        pass

    return run_parallel(with_retry, items, max_workers)

### Exercise 3: Confidence-aware voting
Write `majority_vote(question, context, n=7)` that returns `{"answer": ..., "agreement": ...}` and flags `needs_human=True` when the agreement is below a threshold you choose.
Then test it on one question the abstract answers clearly and one it leaves ambiguous, and compare the agreement levels.

In [ ]:
def majority_vote(question: str, context: str, n: int = 7, threshold: float = 0.7) -> dict:
    # TODO: run n parallel votes, count them, compute agreement and needs_human
    pass


# TODO: try it on two questions, one clear and one ambiguous

## 🏁 Summary & Key Takeaways

- Use parallelization when the calls are **independent**. If B needs A's output, it is a chain.
- **Sectioning**: different specialists, same input, faster and more focused. **Voting**: same agent N times, more reliable answers.
- LLM calls are I/O-bound, so **threads** are the right tool. Prefer `ThreadPoolExecutor` over raw threads.
- Key results by **name** (order is not deterministic), capture **exceptions** through futures, cap `max_workers` and use **backoff** for rate limits.
- The **fan-in** agent needs its own clear job (compare, weigh, decide), and should know when a result is missing.

### What's next?
- **Deep Dive 5: Evaluator-Optimizer**: one agent generates, another critiques, and the loop repeats until the output meets your constraints.